In [60]:
from apriori.parser import parse_csv, parse_transactions
from apriori.utils import _check_next_round

transactions = {
    1: {"Onion", "Potato", "Burger"},
    2: {"Potato", "Burger", "Milk"},
    3: {"Milk", "Beer"},
    4: {"Onion", "Potato", "Milk"},
    5: {"Onion", "Potato", "Burger", "Beer"},
    6: {"Onion", "Potato", "Burger", "Milk", "Beer"},
}

from_csv = parse_csv("./data/apriori_transactions.csv")
from_dict = parse_transactions(transactions)

display(from_csv)

,Beer,Burger,Milk,Onion,Potato
1,0,1,0,1,1
2,0,1,1,0,1
3,1,0,1,0,0
4,0,0,1,1,1
5,1,1,0,1,1
6,1,1,1,1,1


In [61]:
help(_check_next_round)

Help on function _check_next_round in module apriori.utils:

_check_next_round(
    transactions: pandas.DataFrame,
    candidates: collections.abc.Iterable[str | tuple[str, ...]],
    min_support: float
) -> tuple[list[tuple[str, ...]], list[tuple[str, ...]]]
    Keep frequent itemsets and form valid candidates for the next round.



In [62]:
items = from_dict.columns

one_itemset = _check_next_round(from_dict, items, min_support=0.5)

In [63]:
print(one_itemset)

([('Beer',), ('Burger',), ('Milk',), ('Onion',), ('Potato',)], [('Beer', 'Burger'), ('Beer', 'Milk'), ('Beer', 'Onion'), ('Beer', 'Potato'), ('Burger', 'Milk'), ('Burger', 'Onion'), ('Burger', 'Potato'), ('Milk', 'Onion'), ('Milk', 'Potato'), ('Onion', 'Potato')])


In [64]:
candidates = from_dict.columns
frequent_itemsets_by_round = {}
round_number = 1

while len(candidates) > 0:
    frequent_itemsets, candidates = _check_next_round(
        from_dict, candidates, min_support=0.5
    )
    frequent_itemsets_by_round[round_number] = frequent_itemsets
    round_number += 1

In [65]:
list(frequent_itemsets_by_round.keys())[-1]

3

In [66]:
frequent_itemsets_by_round.keys()

dict_keys([1, 2, 3])

In [67]:
from apriori.utils import _support

for k, v in frequent_itemsets_by_round.items():
    print(f"Round {k}: {[(itemset, _support(from_dict, itemset)) for itemset in v]}")

Round 1: [(('Beer',), 0.5), (('Burger',), 0.6666666666666666), (('Milk',), 0.6666666666666666), (('Onion',), 0.6666666666666666), (('Potato',), 0.8333333333333334)]
Round 2: [(('Burger', 'Onion'), 0.5), (('Burger', 'Potato'), 0.6666666666666666), (('Milk', 'Potato'), 0.5), (('Onion', 'Potato'), 0.6666666666666666)]
Round 3: [(('Burger', 'Onion', 'Potato'), 0.5)]


In [68]:
last_round = list(frequent_itemsets_by_round.keys())[-1]

frequent_itemsets_by_round[last_round]

[('Burger', 'Onion', 'Potato')]

In [69]:
def get_subset(frequent_itemsets: dict, round: int | None = None):
    from itertools import combinations

    if round is None:
        if not frequent_itemsets:
            raise ValueError("There are no frequent itemsets.")
        round = list(frequent_itemsets.keys())[-1]
    if round not in frequent_itemsets:
        raise ValueError(f"Round {round} does not exist.")
    subsets = {}
    for itemset in frequent_itemsets[round]:
        subsets[itemset] = [
            subset for size in range(1, len(itemset)) for subset in combinations(itemset, size)
        ]
    return subsets

In [71]:
from apriori.utils import _confidence

def get_association_rules(transactions, subsets, min_confidence):
    if not 0 <= min_confidence <= 1:
        raise ValueError("min_confidence must be between 0 and 1")

    rules = []
    for itemset, antecedents in subsets.items():
        for antecedent in antecedents:
            consequent = tuple(item for item in itemset if item not in antecedent)
            confidence = _confidence(transactions, (antecedent, consequent))
            if confidence >= min_confidence:
                rules.append((antecedent, consequent, confidence))
    return rules

rules = get_association_rules(from_dict, get_subset(frequent_itemsets_by_round), 0.7)
rules

[(('Burger',), ('Onion', 'Potato'), 0.75),
 (('Onion',), ('Burger', 'Potato'), 0.75),
 (('Burger', 'Onion'), ('Potato',), 1.0),
 (('Burger', 'Potato'), ('Onion',), 0.75),
 (('Onion', 'Potato'), ('Burger',), 0.75)]